# 17.2 — NPP-Guard v1 protocol alignment

This notebook freezes a trajectory-level release split, audits sample-id overlap, rebuilds fixed artifacts without touching the locked release test, checks exact batch/API parity, and reports a release benchmark. The milestone-16 severity-extrapolation result remains a separate research stress benchmark and is not a release parity gate.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != 'NPP-Guard':
    PROJECT_ROOT = Path(r'C:/Users/18205/NPP-Guard')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.inference import (
    build_v1_artifacts,
    evaluate_pipeline_parity,
    evaluate_release_benchmark,
    find_behavior_examples,
    run_regression_tests,
    verify_artifact_manifest,
)

manifest = build_v1_artifacts(PROJECT_ROOT)
parity = evaluate_pipeline_parity(PROJECT_ROOT)
benchmark = evaluate_release_benchmark(PROJECT_ROOT)
tests, examples = run_regression_tests(PROJECT_ROOT, parity_result=parity)

result_root = PROJECT_ROOT / 'results'
tests.to_csv(result_root / '17_v1_regression_tests.csv', index=False)
(result_root / '17_v1_example_outputs.json').write_text(json.dumps(examples, ensure_ascii=False, indent=2), encoding='utf-8')
overlap = json.loads((result_root / '17_v1_overlap_audit.json').read_text(encoding='utf-8'))
summary = {
    'model_version': manifest['model_version'],
    'api': ['diagnose_csv(path)', 'diagnose_dataframe(df)'],
    'cli': 'python -m src.inference.cli --input <csv>',
    'decision_policy': manifest['policy'],
    'artifact_manifest': 'artifacts/v1/manifest.json',
    'protocol': manifest['protocol'],
    'release_protocol_alignment': {'status': 'passed', 'protocol': manifest['protocol']['version']},
    'overlap_audit': overlap['assertions'],
    'exact_pipeline_parity': parity,
    'locked_release_benchmark': benchmark,
    'research_stress_benchmark': {
        'source': 'results/16_summary.json and results/16_*',
        'same_protocol_as_release_benchmark': False,
        'strict_parity_gate': False,
        'note': 'Milestone 16 used severity-extrapolation research splits; v1 release uses the permanently locked random/seed_20260921 test cohort.',
    },
    'behavior_examples': {key: value.get('status') for key, value in examples.items() if key in {'accepted_tier_a', 'accepted_loca_severity', 'accepted_non_loca_tier_a', 'requires_review', 'unknown', 'invalid_input'}},
    'regression_tests': {'count': int(len(tests)), 'passed': int(tests['passed'].sum()), 'all_passed': bool(tests['passed'].all())},
    'loca_severity': manifest['loca_severity'],
    'protection_time': manifest['protection_time'],
    'capability_boundary': manifest['capability_gate'],
    'limitations': manifest['policy']['limitations'],
}
(result_root / '17_v1_integration_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

display(tests)
print('Protocol:', manifest['protocol']['version'])
print('Locked release rows:', benchmark['protocol']['release_sample_count'])
print('Exact discrete parity:', parity['discrete_decision_agreement_rate'])
print('Regression:', int(tests['passed'].sum()), '/', len(tests))

,test,passed,status,detail
0,valid_120s_input,True,requires_review,LOCA/1.csv full trajectory
1,missing_required_column,True,invalid_input,removed one strict process variable
2,window_shorter_than_120s,True,invalid_input,LOCA/1.csv truncated at 50 s
3,nan_inf_rejected,True,invalid_input,inserted NaN in a strict variable
4,inf_rejected,True,invalid_input,inserted Inf in a strict variable
5,extra_column_ignored,True,requires_review,extra column is ignored and reported
6,duplicate_time_rejected,True,invalid_input,duplicated TIME point
7,loca_severity_gate_matches_policy,True,requires_review,LOCA severity only runs for accepted LOCA Tier...
8,ood_sample_runs,True,unknown,real LOCA/100.csv severity-shift proxy
9,requires_review_does_not_run_loca_severity,True,requires_review,locked release behavior example


Protocol: v1_release_protocol_17_2
Locked release rows: 101
Exact discrete parity: 1.0
Regression: 18 / 18


In [2]:
assert overlap['assertions']['release_test_disjoint_from_all_fit_sets']
assert verify_artifact_manifest(PROJECT_ROOT / 'artifacts' / 'v1')['all_passed']
assert parity['overall_pass']
assert benchmark['protocol']['release_sample_count'] == 101
assert bool(tests['passed'].all())
assert all(examples[key].get('status') != 'not_observed_in_current_evaluation_set' for key in ['accepted_tier_a', 'accepted_loca_severity', 'requires_review', 'unknown', 'invalid_input'])
assert manifest['policy']['conformal_alpha'] == 0.10
assert manifest['inference_never_retrains'] is True
print('FULL NPP-GUARD V1.2 RELEASE GATE PASSED')

FULL NPP-GUARD V1.2 RELEASE GATE PASSED
